# 22. 개수 피처를 비율 피처로 바꾸기 (H22)

## 배경 (11/11 회의)
- 표기 정규화 후에도 test 환자는 train보다 변이가 약 2배 많음 (환자당 변이 유전자 수 중앙값 14 → 28)
- 15: XGB에서 stats를 빼면 CV 0.4161 → 0.3579 → 모델은 "변이가 몇 개냐"를 아주 중요하게 봄
- 그런데 지금 stats는 거의 다 **개수** (미스센스·넌센스·프레임시프트 개수 log), 비율은 침묵 비율 하나뿐
- test는 변이가 2배라서 같은 암이어도 개수가 2배로 나옴 → train에서 배운 기준이 어긋남
- 그래서 개수를 **비율**로 바꾸고, 전체 변이 수는 log로 하나만 남김 (예: 미스센스 6 / 전체 10 → 60%, 12 / 20 → 60%)
- 지금까지 LB가 오른 실험(09 모델 교체, 10 튜닝, 14 섞기)은 모두 "덜 외우게 하는" 쪽이었음

## 실험 순서 (한 번에 하나씩 쌓기)
| 번호 | 내용 |
| :--- | :--- |
| **22a** | stats 유형별 개수 → 비율 + 전체 변이 수 log |
| 22b | 22a + 미스센스 아미노산 치환 종류 비율 (R→H 등, 상위 N개는 fold 학습 부분에서만) |
| 22c | 22b + 아미노산 성질 그룹 변화 비율 |

## 가설
- H22a: stats를 비율판(`ratio`)으로 바꾸면 test 변이 수 증가에 덜 흔들려 LB가 오른다 (CV는 비슷하거나 약간 낮을 수 있음)

## 검증 방법
- 모델: 10 최적 설정 XGB 고정 (depth 3, lr 0.1, 200그루, colsample 0.3), 데이터: `load_data()` (정규화)
- 비교: 기준 `any + stats` vs 22a `any + ratio` (같은 fold, 피처만 다름)
  - `ratio` = 유형별 변이 비율 7개(유형 수 / 전체 변이 수) + 전체 변이 수 log1p → `src/features.py`의 `mutation_features()`
  - 한 환자 안에서만 계산 → Data Leakage 아님
- 채택 기준 (11/11 회의): **5 fold 중 4개 이상 오르면** 단톡 공유 → 제출 후보, 비교 기준 CV 0.4161 / LB 0.32466(14)
- 실행 환경: 이 노트북은 macOS M1 → XGB 점수는 데스크탑(Windows)과 ±0.004 다를 수 있음 (같은 노트북 안의 비교는 같은 조건)
- 제출 파일은 데스크탑(Windows)에서 실행할 때만 저장 (팀 규칙)

#### 준비: 데이터, 피처, 10 최적 설정

In [1]:
import sys; sys.path.append('..')
import platform, time, numpy as np, pandas as pd, xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED
from src.data import load_data, get_gene_cols
from src.features import mutation_features
from src.validation import seed_everything, encode_target, get_folds, run_cv, save_submission

print(f'{platform.system()} | CPU {platform.machine()} | xgboost {xgb.__version__}')
seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
folds = get_folds(y)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)

class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

BEST = dict(max_depth=3, learning_rate=0.1, n_estimators=200, colsample_bytree=0.3,
            tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1)
print('stats 열:', list(F_tr['stats'].columns))
print('ratio 열:', list(F_tr['ratio'].columns))

Darwin | CPU arm64 | xgboost 3.4.1
stats 열: ['n_silent_log', 'n_missense_log', 'n_nonsense_log', 'n_frameshift_log', 'n_inframe_indel_log', 'n_stop_loss_log', 'n_other_log', 'n_gene_log', 'silent_ratio']
ratio 열: ['r_silent', 'r_missense', 'r_nonsense', 'r_frameshift', 'r_stop_loss', 'r_other', 'n_mut_log']


#### 치환 함수 확인 (PR #29 리뷰 요청)
- 22b·22c에서 쓰는 `aa_substitution()`, `aa_group_change()`가 어떤 토큰을 처리하지 못하는지 유형별로 확인
  - 처리 못 하는 것(None)은 단일 아미노산 치환이 아닌 토큰: frameshift, in-frame indel, 여러 아미노산 치환(`1529_1530AL>AL`)
  - test 파싱률이 train보다 낮은 이유가 이 유형들 때문인지 확인 (test 값은 관찰용, 피처 선택에 쓰지 않음)
- 침묵 변이는 원래·바뀐 아미노산이 같아서 성질 그룹도 항상 같게 나옴
  → 22b·22c 비율은 **미스센스만** 걸러서 계산 (`mutation_type(t) == 'missense'`)

In [2]:
import collections
from src.features import aa_substitution, aa_group_change, mutation_type

rows = {}
for name, df in [('train', train), ('test', test)]:
    V = df[genes].to_numpy()
    T = [t for c in V[V != 'WT'] for t in c.split()]
    un = [t for t in T if aa_substitution(t) is None]          # 단일 치환으로 처리 못 한 토큰
    cnt = collections.Counter('여러 아미노산 치환(>)' if '>' in t else mutation_type(t) for t in un)
    cnt['합계'] = len(un)
    cnt['파싱 안 됨 비율(%)'] = round(len(un) / len(T) * 100, 1)
    sil = [aa_group_change(t) for t in T if mutation_type(t) == 'silent']
    cnt['침묵: 같은 그룹 비율(%)'] = round(sum(a.split('>')[0] == a.split('>')[1] for a in sil if a) / len(sil) * 100, 1)
    rows[name] = cnt
pd.DataFrame(rows).fillna(0)

,train,test
frameshift,9753.0,13649.0
여러 아미노산 치환(>),233.0,40.0
inframe_indel,3.0,3501.0
합계,9989.0,17191.0
파싱 안 됨 비율(%),4.0,8.4
침묵: 같은 그룹 비율(%),99.9,100.0
other,0.0,1.0


- 결과: 처리 못 한 토큰은 대부분 frameshift(train 9,753 / test 13,649)와 test에만 있는 in-frame indel(3,501), 여러 아미노산 치환은 233 / 40개, other는 test 1개(`M391`)
  - test 파싱률이 낮은 건 in-frame indel 때문이며 단일 치환이 아니라 None이 의도한 동작
- 침묵의 같은 그룹 비율 99.9% / 100% → 22b·22c는 미스센스만 사용

#### in-frame indel 표기 확인
- in-frame indel이 train 3개 / test 3,501개로 차이가 커서, 종결코돈(`X` → `*`)처럼 **표기 차이**인지 확인
  - 표기 차이라면: test indel과 같은 유전자·위치에 train은 다른 표기(예: `>` 연속 치환)로 적혀 있어야 함
- 이대로 비율 분모에 넣으면 test에서만 미스센스 비율 등이 낮아짐 → 22 목적(변이 수 차이에 덜 흔들리게)과 반대
- test 값은 원인 확인용, 피처 선택에는 쓰지 않음

In [3]:
import re
POS = re.compile(r'\d+')
tr_tok = collections.defaultdict(list)          # train: 유전자 → 토큰 목록
for gene in genes:
    for v in train[gene][train[gene] != 'WT']:
        tr_tok[gene].extend(v.split())

indel = [(gene, t) for gene in genes for v in test[gene][test[gene] != 'WT'] for t in v.split()
         if mutation_type(t) == 'inframe_indel']
shape = lambda t: re.sub(r'[A-Z*]', 'a', re.sub(r'\d+', 'N', t))
print('test in-frame indel', len(indel), '| 표기 모양 상위:', collections.Counter(shape(t) for _, t in indel).most_common(5))

same_pos, examples = collections.Counter(), []
for gene, t in indel:
    p = int(POS.search(t).group())
    hit = [x for x in tr_tok.get(gene, []) if POS.search(x) and int(POS.search(x).group()) == p]
    same_pos[collections.Counter(map(mutation_type, hit)).most_common(1)[0][0] if hit else 'train에 같은 위치 없음'] += 1
    if hit and len(examples) < 8:
        examples.append({'유전자': gene, 'test indel': t, '같은 위치 train 토큰': ', '.join(hit[:3])})
print('같은 유전자·위치의 train 토큰 유형:', dict(same_pos))
print('train의 del/ins 토큰:', [t for L in tr_tok.values() for t in L if re.search('del|ins|dup', t)])
pd.DataFrame(examples)

test in-frame indel 3501 | 표기 모양 상위: [('aadel', 974), ('aa_aadel', 888), ('aa_aainsa', 448), ('aa_aadelinsa', 328), ('aa_aainsaa', 179)]
같은 유전자·위치의 train 토큰 유형: {'train에 같은 위치 없음': 3139, 'silent': 80, 'missense': 242, 'nonsense': 15, 'frameshift': 25}
train의 del/ins 토큰: ['R649del', '490del', '249del']


,유전자,test indel,같은 위치 train 토큰
0,ABCG2,K360del,K360K
1,ACADS,K331_D332insK,A331T
2,ACOX2,Q488_R489insFFND,Q488H
3,ACTA1,S237del,S237F
4,APBB1,D166_E167insD,D166E
5,APC,K1303_S1305del,Q1303*
6,APC,S1454_N1455insVWRFLRDLELEIPFDP,"K1454fs, K1454fs"
7,APC,S1419_N1420insVWRFLRDLELEIPFDP,S1419N


- 결과: test indel 3,501개 중 **3,139개(90%)는 train에 같은 유전자·위치의 변이가 아예 없음**
  - 같은 위치가 있는 경우도 `K360del` ↔ `K360K`, `K331_D332insK` ↔ `A331T`처럼 **다른 변이**(같은 코돈의 단일 치환)이지 같은 변이의 다른 표기가 아님
  - train에는 `>` 연속 치환 233개, del 3개(`R649del`, `490del`, `249del`)뿐 → 다른 표기로 바꿔 적은 흔적이 없음
- 결론: 표기 차이의 근거가 없음 → train 데이터에 in-frame indel이 거의 빠져 있는 것으로 보임
  → 리뷰 제안대로 **`ratio`의 분모와 열에서 inframe_indel 제외** (`src/features.py`)

#### 왜 비율인가: train vs test 평균 (관찰용)
- 개수 피처는 test에서 크게 커지고, 비율 피처는 덜 변하는지 확인
- test 값은 **피처를 고르거나 기준을 정하는 데 쓰지 않음** (팀 규칙: test는 형식 확인·관찰만)

In [4]:
def shift(name):
    a, b = F_tr[name].mean(), F_te[name].mean()
    return pd.DataFrame({'train 평균': a.round(3), 'test 평균': b.round(3), 'test − train': (b - a).round(3)})

pd.concat({'stats (개수 log 중심)': shift('stats'), 'ratio (비율 + 전체 변이 수 log)': shift('ratio')})

train 평균  test 평균  test − train
stats (개수 log 중심)        n_silent_log            1.528    2.039         0.511
                         n_missense_log          2.389    2.891         0.502
                         n_nonsense_log          0.617    0.768         0.151
                         n_frameshift_log        0.564    0.852         0.289
                         n_inframe_indel_log     0.000    0.394         0.394
                         n_stop_loss_log         0.000    0.032         0.032
                         n_other_log             0.000    0.000         0.000
                         n_gene_log              2.756    3.299         0.543
                         silent_ratio            0.225    0.239         0.014
ratio (비율 + 전체 변이 수 log) r_silent                0.225    0.242         0.018
                         r_missense              0.645    0.641        -0.004
                         r_nonsense              0.051    0.043        -0.007
                         r_frameshift            0.064    0.061        -0.003
                         r_stop_loss             0.000    0.000         0.000
                         r_other                 0.000    0.000         0.000
                         n_mut_log               2.780    3.303         0.523

#### 22a: 기준(any + stats) vs any + ratio
- 같은 fold로 피처만 바꿔 비교, fold별 점수로 "5 fold 중 4개 이상" 확인

In [5]:
SETS = {
    '기준: any + stats': (pd.concat([F_tr['any'], F_tr['stats']], axis=1), pd.concat([F_te['any'], F_te['stats']], axis=1)),
    '22a: any + ratio':  (pd.concat([F_tr['any'], F_tr['ratio']], axis=1), pd.concat([F_te['any'], F_te['ratio']], axis=1)),
}
R = {}
for name, (X, X_te) in SETS.items():
    t = time.time()
    R[name] = run_cv(lambda: BalancedXGB(**BEST), X, y, X_te, folds=folds, verbose=False)
    print(f"{name}: CV {R[name]['cv_mean']:.4f} ± {R[name]['cv_std']:.4f} (OOF {R[name]['oof_score']:.4f}, {time.time() - t:.0f}s)")

base = np.array(R['기준: any + stats']['fold_scores'])
fold_df = pd.DataFrame({k: np.round(v['fold_scores'], 4) for k, v in R.items()}, index=[f'fold {i}' for i in range(5)])
fold_df['차이 (22a − 기준)'] = (fold_df['22a: any + ratio'] - fold_df['기준: any + stats']).round(4)
print('22a가 기준보다 높은 fold:', int((np.array(R['22a: any + ratio']['fold_scores']) > base).sum()), '/ 5')
fold_df

기준: any + stats: CV 0.4125 ± 0.0109 (OOF 0.4131, 168s)
22a: any + ratio: CV 0.4029 ± 0.0025 (OOF 0.4037, 162s)
22a가 기준보다 높은 fold: 1 / 5


,기준: any + stats,22a: any + ratio,차이 (22a − 기준)
fold 0,0.4082,0.4011,-0.0071
fold 1,0.4230,0.4066,-0.0164
fold 2,0.4278,0.4052,-0.0226
fold 3,0.4023,0.4000,-0.0023
fold 4,0.4014,0.4018,0.0004


#### test 예측 분포 (관찰용, 판단 기준 아님)
- 개수 피처가 test에서 커지면 변이 많은 암(COAD 등)으로 쏠리는지

In [6]:
pd.DataFrame({k: pd.Series(le.classes_[v['test_proba'].argmax(1)]).value_counts(normalize=True)
              for k, v in R.items()}).fillna(0).sort_values('22a: any + ratio', ascending=False).head(6).mul(100).round(1).astype(str) + '%'

,기준: any + stats,22a: any + ratio
COAD,10.9%,13.5%
STES,13.6%,12.6%
BRCA,4.9%,4.8%
OV,4.9%,4.7%
TGCT,5.5%,4.7%
LUSC,5.1%,4.5%


#### 22b: 22a + 미스센스 아미노산 치환 종류 비율
- 환자마다 미스센스를 R→H, E→K처럼 **바뀐 종류별로 세서 미스센스 중 비율**로 만듦 (`aa_substitution()`, 한 환자 안에서만 계산)
  - `mutation_type(t) == 'missense'`인 토큰만 사용 (침묵 R>R·넌센스 R>* 제외), 여러 아미노산 치환(447_448MD>IN)은 단일 치환이 아니라 제외
  - 담배·자외선·DNA 복구 고장처럼 암의 원인마다 잘 생기는 치환 종류가 다름 (11번 핫스팟과 달리 유전자·위치는 빼고 "바뀐 종류"만)
- 치환 종류는 train에 242가지 → **상위 N개만** 열로 사용
  - 상위 N개 목록은 여러 환자의 통계 → `TopNSubs`가 **fold 학습 부분에서만** 골라 Pipeline 안에서 적용 (Data Leakage 방지)
  - N ∈ {30, 100} 비교 (train 기준 상위 30개 ≈ 미스센스의 44%, 100개 ≈ 88%)

In [7]:
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from src.config import WT
from src.features import aa_substitution, mutation_type

def sub_ratio_matrix(df):
    # 환자별 미스센스 치환 종류 비율 (전체 종류, 한 환자 안에서만 계산 → 누수 아님)
    rows = []
    for row in df[genes].to_numpy():
        subs = (aa_substitution(t) for v in row if v != WT for t in v.split() if mutation_type(t) == 'missense')
        c = Counter(s for s in subs if s is not None)   # 여러 아미노산 치환(447_448MD>IN)은 단일 치환이 아니라 제외
        tot = sum(c.values())
        rows.append({k: n / tot for k, n in c.items()} if tot else {})
    M = pd.DataFrame(rows, index=df.index).fillna(0).astype(np.float32)
    M.columns = ['sub__' + k.replace('>', '_') for k in M.columns]   # R>H → sub__R_H
    return M

S_tr = sub_ratio_matrix(train)
S_te = sub_ratio_matrix(test).reindex(columns=S_tr.columns, fill_value=0)   # 열은 train 기준 (test에만 있는 종류는 버림)
print('치환 종류(열) 수:', S_tr.shape[1])

class TopNSubs(BaseEstimator, TransformerMixin):
    # fold 학습 부분에서 비율 합이 큰 상위 n개 치환 열만 남김. 목록은 fit에서만 정함
    def __init__(self, n=30):
        self.n = n
    def fit(self, X, y=None):
        sub = [c for c in X.columns if c.startswith('sub__')]
        keep = X[sub].sum().sort_values(ascending=False).index[:self.n]
        self.drop_ = [c for c in sub if c not in set(keep)]
        self.kept_ = list(keep)
        return self
    def transform(self, X):
        return X.drop(columns=self.drop_)

X22a, X22a_te = SETS['22a: any + ratio']
X22b, X22b_te = pd.concat([X22a, S_tr], axis=1), pd.concat([X22a_te, S_te], axis=1)
for n in [30, 100]:
    name = f'22b: 22a + 치환 비율 상위 {n}'
    t = time.time()
    R[name] = run_cv(lambda: make_pipeline(TopNSubs(n), BalancedXGB(**BEST)), X22b, y, X22b_te, folds=folds, verbose=False)
    print(f"{name}: CV {R[name]['cv_mean']:.4f} ± {R[name]['cv_std']:.4f} (OOF {R[name]['oof_score']:.4f}, {time.time() - t:.0f}s)")
print('fold 0에서 고른 상위 10개:', [c[5:].replace('_', '>') for c in R['22b: 22a + 치환 비율 상위 30']['models'][0][0].kept_[:10]])

치환 종류(열) 수: 241
22b: 22a + 치환 비율 상위 30: CV 0.4437 ± 0.0050 (OOF 0.4446, 166s)
22b: 22a + 치환 비율 상위 100: CV 0.4487 ± 0.0065 (OOF 0.4497, 174s)
fold 0에서 고른 상위 10개: ['R>H', 'E>K', 'R>C', 'R>Q', 'A>T', 'A>V', 'R>W', 'D>N', 'P>L', 'P>S']


#### 22c: 22b + 아미노산 성질 그룹 변화 비율
- 미스센스마다 "어느 성질 그룹에서 어느 그룹으로" 바뀌었는지(`aa_group_change()`)를 세서 **미스센스 중 비율**로 만듦
  - 그룹: 소수성 / 친수성 / 양전하 / 음전하 / 글리신 / 프롤린 (11/11 회의, 프라탁신 논문)
  - 논문 결론: 변이의 영향은 **위치**와 **성질 변화**로 결정 → 위치(11 핫스팟)는 LB에서 효과가 없었으니 성질 변화를 봄
- 22b와 같은 필터: `mutation_type(t) == 'missense'`인 토큰만 (침묵 R>R이 positive>positive로 섞이지 않게)
- 열은 6그룹 × 6그룹 = **36개 고정** (도메인 지식으로 정한 목록이라 fold 안에서 고를 필요 없음, train에 없는 조합은 0)
- 22b의 N은 22b 두 설정 중 CV 평균이 높은 쪽 사용 (CV 기준 선택)

In [8]:
from src.features import AA_GROUPS, aa_group_change
GROUPS6 = [g for g in AA_GROUPS if g != 'stop']
GROUP_COLS = [f'grp__{a}_{b}' for a in GROUPS6 for b in GROUPS6]   # 36개 고정

def group_ratio_matrix(df):
    # 환자별 미스센스 성질 그룹 변화 비율 (한 환자 안에서만 계산 → 누수 아님)
    rows = []
    for row in df[genes].to_numpy():
        chg = (aa_group_change(t) for v in row if v != WT for t in v.split() if mutation_type(t) == 'missense')
        c = Counter(x for x in chg if x is not None)
        tot = sum(c.values())
        rows.append({'grp__' + k.replace('>', '_'): n / tot for k, n in c.items()} if tot else {})
    return pd.DataFrame(rows, index=df.index).reindex(columns=GROUP_COLS, fill_value=0).fillna(0).astype(np.float32)

G_tr, G_te = group_ratio_matrix(train), group_ratio_matrix(test)
print('성질 그룹 변화 열:', G_tr.shape[1], '| train에서 한 번이라도 나온 조합:', int((G_tr.sum() > 0).sum()))

best_n = max([30, 100], key=lambda n: R[f'22b: 22a + 치환 비율 상위 {n}']['cv_mean'])
name = f'22c: 22b(상위 {best_n}) + 성질 그룹 변화 비율'
X22c, X22c_te = pd.concat([X22b, G_tr], axis=1), pd.concat([X22b_te, G_te], axis=1)
t = time.time()
R[name] = run_cv(lambda: make_pipeline(TopNSubs(best_n), BalancedXGB(**BEST)), X22c, y, X22c_te, folds=folds, verbose=False)
print(f"{name}: CV {R[name]['cv_mean']:.4f} ± {R[name]['cv_std']:.4f} (OOF {R[name]['oof_score']:.4f}, {time.time() - t:.0f}s)")

성질 그룹 변화 열: 36 | train에서 한 번이라도 나온 조합: 33
22c: 22b(상위 100) + 성질 그룹 변화 비율: CV 0.4448 ± 0.0027 (OOF 0.4454, 182s)


#### 22a·22b·22c 비교: 기준 대비 / 22a 대비 (fold별)


In [9]:
a = np.array(R['22a: any + ratio']['fold_scores'])
rows = []
for name, r in R.items():
    fs = np.array(r['fold_scores'])
    rows.append({'피처': name, 'CV': round(fs.mean(), 4), 'std': round(fs.std(), 4), 'OOF': round(r['oof_score'], 4),
                 '기준 대비': round(fs.mean() - base.mean(), 4), '기준보다 높은 fold': f'{int((fs > base).sum())}/5',
                 '22a 대비': round(fs.mean() - a.mean(), 4), '22a보다 높은 fold': f'{int((fs > a).sum())}/5'})
pd.DataFrame(rows)

,피처,CV,std,OOF,기준 대비,기준보다 높은 fold,22a 대비,22a보다 높은 fold
0,기준: any + stats,0.4125,0.0109,0.4131,0.0000,0/5,0.0096,4/5
1,22a: any + ratio,0.4029,0.0025,0.4037,-0.0096,1/5,0.0000,0/5
2,22b: 22a + 치환 비율 상위 30,0.4437,0.0050,0.4446,0.0311,5/5,0.0407,5/5
3,22b: 22a + 치환 비율 상위 100,0.4487,0.0065,0.4497,0.0362,5/5,0.0458,5/5
4,22c: 22b(상위 100) + 성질 그룹 변화 비율,0.4448,0.0027,0.4454,0.0323,5/5,0.0419,5/5


#### 22b·22c test 예측 분포 (관찰용, 판단 기준 아님)
- 치환 비율을 넣으면 test 예측이 어느 암종으로 움직이는지


In [10]:
pd.DataFrame({k: pd.Series(le.classes_[v['test_proba'].argmax(1)]).value_counts(normalize=True)
              for k, v in R.items()}).fillna(0).sort_values('기준: any + stats', ascending=False).head(8).mul(100).round(1).astype(str) + '%'


,기준: any + stats,22a: any + ratio,22b: 22a + 치환 비율 상위 30,22b: 22a + 치환 비율 상위 100,22c: 22b(상위 100) + 성질 그룹 변화 비율
STES,13.6%,12.6%,17.9%,21.8%,20.0%
COAD,10.9%,13.5%,13.4%,12.4%,13.6%
TGCT,5.5%,4.7%,4.1%,2.2%,2.2%
LUSC,5.1%,4.5%,3.5%,1.8%,1.9%
OV,4.9%,4.7%,4.7%,4.3%,4.6%
BRCA,4.9%,4.8%,5.2%,5.3%,5.5%
BLCA,4.4%,3.6%,1.4%,1.2%,0.9%
UCEC,3.9%,4.1%,3.8%,3.8%,3.6%


#### 제출 파일 (데스크탑에서만 저장)
- 22 설정 중 채택 기준(기준보다 5 fold 중 4개 이상 상승)을 넘는 것 가운데 CV 최고를 저장
- 팀 규칙: LB 비교용 제출 파일은 설아님 Windows 데스크탑에서만 생성 → 다른 컴퓨터에서는 저장하지 않음


In [11]:
# 22 설정 중 '기준보다 4개 이상 fold에서 높은' 것 가운데 CV 최고를 저장 (11/11 채택 기준)
cands = {k: v for k, v in R.items() if k.startswith('22') and (np.array(v['fold_scores']) > base).sum() >= 4}
if not cands:
    print('채택 기준(5 fold 중 4개 이상)을 넘는 22 설정 없음 → 저장 안 함')
else:
    best = max(cands, key=lambda k: cands[k]['cv_mean'])
    print('저장 대상:', best)
    tag = {'22b: 22a + 치환 비율 상위 100': 'sub100', '22b: 22a + 치환 비율 상위 30': 'sub30'}.get(best, 'grp')
    if platform.system() == 'Windows':
        save_submission(submission, cands[best]['test_proba'], le, name=f'xgb_d3_cs0.3_ratio22_{tag}', cv_score=cands[best]['cv_mean'])
    else:
        print(f'{platform.system()}에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)')


저장 대상: 22b: 22a + 치환 비율 상위 100
Darwin에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)


#### 22b + LR 섞기 (제출 후보 C)
- 14(XGB 0.7 + LR 0.3, **LB 0.32466 최고**)와 같은 방식에서 **XGB만 22b(상위 100)로 교체**
  - LR은 14와 같은 설정 (any, C=0.1, balanced), 비율 0.7 / 0.3도 14와 같음 (다시 고르지 않음)
- 22b 단독은 test의 APC 절단 환자를 STES로 많이 옮김(COAD 63% → 47%) → 유전자 0/1을 그대로 보는 LR을 섞어 COAD 신호를 지키는지 확인
- 비교: 14 조합(기준 XGB 0.7 + LR 0.3)과 같은 fold로 짝 비교
- 제출 파일은 Windows(데스크탑)에서만 저장

In [12]:
from sklearn.linear_model import LogisticRegression
from src.validation import macro_f1
from src.features import mutation_type as _mt

r_lr = run_cv(lambda: LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced', random_state=SEED),
              F_tr['any'].astype('float32'), y, F_te['any'].astype('float32'), folds=folds, verbose=False)
X22B = R['22b: 22a + 치환 비율 상위 100']
BL = {
    '14: 기준 XGB 0.7 + LR 0.3':  (0.7 * R['기준: any + stats']['oof_proba'] + 0.3 * r_lr['oof_proba'],
                                  0.7 * R['기준: any + stats']['test_proba'] + 0.3 * r_lr['test_proba']),
    '22b 단독':                    (X22B['oof_proba'], X22B['test_proba']),
    'C: 22b XGB 0.7 + LR 0.3':     (0.7 * X22B['oof_proba'] + 0.3 * r_lr['oof_proba'],
                                  0.7 * X22B['test_proba'] + 0.3 * r_lr['test_proba']),
}
fs = lambda P: np.array([macro_f1(y[va], P[va].argmax(1)) for _, va in folds])
b14 = fs(BL['14: 기준 XGB 0.7 + LR 0.3'][0])
apc_te = test['APC'].map(lambda v: v != WT and any(_mt(t) in ('nonsense', 'frameshift') for t in v.split())).to_numpy()
coad, stes = list(le.classes_).index('COAD'), list(le.classes_).index('STES')
rows = []
for name, (po, pt) in BL.items():
    s, pred = fs(po), pt.argmax(1)
    rows.append({'조합': name, 'CV': round(s.mean(), 4), 'std': round(s.std(), 4), 'OOF': round(macro_f1(y, po.argmax(1)), 4),
                 '14보다 높은 fold': f'{int((s > b14).sum())}/5',
                 'test COAD 예측': f'{np.mean(pred == coad):.1%}', 'test STES 예측': f'{np.mean(pred == stes):.1%}',
                 'test APC 절단 → COAD': f'{np.mean(pred[apc_te] == coad):.0%}'})
pd.DataFrame(rows)

,조합,CV,std,OOF,14보다 높은 fold,test COAD 예측,test STES 예측,test APC 절단 → COAD
0,14: 기준 XGB 0.7 + LR 0.3,0.4187,0.0065,0.4190,0/5,16.3%,11.7%,63%
1,22b 단독,0.4487,0.0065,0.4497,5/5,12.4%,21.8%,47%
2,C: 22b XGB 0.7 + LR 0.3,0.4557,0.0065,0.4565,5/5,15.0%,20.1%,57%


In [13]:
po, pt = BL['C: 22b XGB 0.7 + LR 0.3']
if platform.system() == 'Windows':
    save_submission(submission, pt, le, name='blend_ratio22sub100_0.7_LR0.3', cv_score=macro_f1(y, po.argmax(1)))   # 파일명 숫자 = OOF (14와 같은 기준)
else:
    print(f'{platform.system()}에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)')

Darwin에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)


## 결과 (macOS M1)

**in-frame indel 확인**: test 3,501개 중 90%는 train에 같은 유전자·위치 변이가 없음 → 표기 차이 근거 없음 → `ratio`의 분모·열에서 제외

**CV 비교 (같은 fold, 피처만 다름)**

| 피처 | CV (± std) | OOF | 기준 대비 | 기준보다 높은 fold | 22a 대비 |
| :--- | :-: | :-: | :-: | :-: | :-: |
| 기준: any + stats | 0.4125 ± 0.0109 | 0.4131 | — | — | +0.0096 |
| 22a: any + ratio | 0.4029 ± 0.0025 | 0.4037 | −0.0096 | 1/5 | — |
| 22b: 22a + 치환 비율 상위 30 | 0.4437 ± 0.0050 | 0.4446 | +0.0311 | 5/5 | +0.0407 |
| **22b: 22a + 치환 비율 상위 100** | **0.4487 ± 0.0065** | **0.4497** | **+0.0362** | **5/5** | **+0.0458** |
| 22c: 22b(상위 100) + 성질 그룹 변화 | 0.4448 ± 0.0027 | 0.4454 | +0.0323 | 5/5 | +0.0419 |

- 참고: 데스크탑 기준 CV는 0.4161 (이 환경은 XGB가 0.004 낮게 나옴)
- 22a(비율만)는 기준보다 낮지만, **치환 종류 비율을 더하면 크게 오름** (+0.036, 5/5 fold, std의 3배 이상)
- 상위 N은 30 → 100으로 늘릴수록 좋음 (치환 종류는 미스센스 중 비율이라 드문 종류도 정보가 있음)
- 22c(성질 그룹 변화)는 22b보다 −0.004 → 치환 종류(20×19)에 이미 성질 정보가 들어 있어, 그룹으로 묶은 열이 새 정보를 보태지 못한 것으로 보임
- fold 0에서 고른 상위 치환: R>H, E>K, R>C, R>Q, A>T, A>V, R>W, D>N, P>L, P>S (C>T 계열이 많음)

**test 예측 분포 (관찰용, 22b 상위 100 별도 실행 확인)**
- STES 21.8% (기준 13.6%, train 6.1%), COAD 12.4% (기준 10.9%)
- test에는 T>P, L>R, Q>P, H>P, F>V, N>T 비율이 train보다 높음 → 모두 A>C / T>G 계열 염기 변화
  - 이 계열은 식도·위암(STES)에 특징적인 변이 지문(COSMIC SBS17로 알려짐, 도메인 지식)과 같은 방향 → test에 STES가 실제로 많을 수도, 다른 원인일 수도 있음 → **LB로만 판단 가능**
- test 값은 관찰만 했고, 피처 선택·기준에는 쓰지 않음

**22b + LR 섞기 (후보 C) — 14와 같은 방식에서 XGB만 22b로 교체**

| 조합 | CV (± std) | OOF | 14보다 높은 fold | test COAD | test STES | test APC 절단 → COAD |
| :--- | :-: | :-: | :-: | :-: | :-: | :-: |
| 14: 기준 XGB 0.7 + LR 0.3 (LB 0.32466) | 0.4187 ± 0.0065 | 0.4190 | — | 16.3% | 11.7% | 63% |
| 22b 단독 | 0.4487 ± 0.0065 | 0.4497 | 5/5 | 12.4% | 21.8% | 47% |
| **C: 22b XGB 0.7 + LR 0.3** | **0.4557 ± 0.0065** | **0.4565** | **5/5** | 15.0% | 20.1% | 57% |

**COAD 예측 분석 (관찰용, 제출 판단 참고)**
- OOF에서는 COAD를 잘 맞힘: 예측 비율 3.6~4.0% (실제 3.6%), COAD F1 14 0.657 → 22b 0.698 / C 0.695
- test의 COAD 예측 중 **92~93%가 APC 절단 변이 환자** (모든 조합 공통)
- APC 절단 변이 환자: train 3.5%(218명) → **test 24.2%(617명), 약 7배**, train에서 이들의 71%가 COAD
  → train 비율대로면 test COAD는 약 440명인데 예측은 315~420명 → test COAD 예측이 높은 건 **과예측보다 test에 대장암 같은 환자가 실제로 많을 가능성** (11/10 회의록 추정과 같음)
- 진짜 위험은 **22b가 이 APC 절단 환자를 STES로 옮기는 것** (COAD 63% → 47%, STES 18% → 43%)
  - test에 많은 A>C / T>G 계열 치환(식도·위암 지문) 때문 → 실제 위·식도암이면 22b가 맞고, 대장암이면 COAD를 놓침 → LB로만 판단 가능
  - C는 유전자 0/1을 그대로 보는 LR이 섞여 중간 (COAD 57%)

## 결론
- **22b(상위 100) 채택 기준 통과**: 기준 대비 CV +0.036, 5 fold 모두 상승
- **후보 C(22b XGB 0.7 + LR 0.3)가 CV 최고** (0.4557, 14 대비 +0.037, 5/5 fold)
- 22c는 기준은 넘지만 22b보다 낮아 제외
- ⚠️ 22b는 test 예측이 14와 37~41% 다름 → LB가 크게 움직일 수 있음 (오르든 내리든)
  - 지금까지 "피처를 더해 오른 CV"는 LB로 안 넘어간 적이 많음(07 E, 11, 16) → 이번엔 비율 피처라 변이 수 증가에 덜 흔들리지만 STES 이동이 있어 LB 확인 필요
- **제출 추천**
  1. **C: 22b XGB 0.7 + LR 0.3** (1순위): 최고 LB(14) 방식 그대로, COAD 신호를 LR이 일부 지킴
  2. B: 22b 단독: 피처 효과만 확인용 (10 + 정규화 0.3202와 비교), STES 이동 위험이 가장 큼
- 제출 파일: 데스크탑에서 이 노트북 실행 → `blend_ratio22sub100_0.7_LR0.3`(C), `xgb_d3_cs0.3_ratio22_sub100`(B) 저장
- LB: C = (제출 후 기입), B = (제출 후 기입) / 비교 기준 14 = 0.32466
